In [30]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [31]:
X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2)

# Because NN are scale sensitive need to scale data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
# want to learn the parameters that are learned from the training set so just transform
X_test_scaled = scaler.transform(X_test)

In [32]:
X_train_scaled_tensor = torch.from_numpy(X_train_scaled).float()
X_test_scaled_tensor = torch.from_numpy(X_test_scaled).float()
y_train_tensor = torch.from_numpy(y_train).float().unsqueeze(1)
y_test_tensor = torch.from_numpy(y_test).float().unsqueeze(1)

In [33]:
train_dataset = TensorDataset(X_train_scaled_tensor, y_train_tensor)

In [34]:
X_train_scaled_tensor.shape

torch.Size([455, 30])

In [35]:
# if unsure of batch size check shape
train_loader = DataLoader(train_dataset,batch_size=32, shuffle=True)

In [36]:
class BCNet(nn.Module):

    def __init__(self):
        super(BCNet, self).__init__()

        self.fc1 = nn.Linear(30,64)
        self.fc2 = nn.Linear(64,32)
        self.fc3 = nn.Linear(32,1)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.sigmoid(self.fc3(x))
        
        return x

In [37]:
model = BCNet()

In [38]:
# loss fn binary cross entropy loss
criterion = nn.BCELoss()
optimiser = optim.Adam(model.parameters(), lr=0.001)

In [39]:
epochs = 20

for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for x_batch, y_batch in train_loader:
        optimiser.zero_grad()

        preds = model(x_batch)
        loss = criterion(preds, y_batch)

        loss.backward()
        optimiser.step()

        running_loss += loss.item()

        print(f'Epoch {epoch+1}: Loss was {running_loss / len(train_loader)}')

Epoch 1: Loss was 0.046582849820454915
Epoch 1: Loss was 0.09460426966349283
Epoch 1: Loss was 0.14006670316060385
Epoch 1: Loss was 0.18472825686136882
Epoch 1: Loss was 0.22874223391215007
Epoch 1: Loss was 0.2727153261502584
Epoch 1: Loss was 0.3158947984377543
Epoch 1: Loss was 0.3580654740333557
Epoch 1: Loss was 0.4000187277793884
Epoch 1: Loss was 0.4413677414258321
Epoch 1: Loss was 0.48154954115549725
Epoch 1: Loss was 0.5216585477193196
Epoch 1: Loss was 0.5613251129786173
Epoch 1: Loss was 0.6001205921173096
Epoch 1: Loss was 0.6374760071436564
Epoch 2: Loss was 0.03716103633244832
Epoch 2: Loss was 0.07254433234532674
Epoch 2: Loss was 0.10776143471399943
Epoch 2: Loss was 0.14423277775446575
Epoch 2: Loss was 0.17654956181844075
Epoch 2: Loss was 0.21029704014460246
Epoch 2: Loss was 0.2413155992825826
Epoch 2: Loss was 0.2743812362353007
Epoch 2: Loss was 0.30935144821802774
Epoch 2: Loss was 0.34029101928075156
Epoch 2: Loss was 0.36916841665903727
Epoch 2: Loss was 0.39

In [41]:
with torch.no_grad():
    model.eval()

    preds = model(X_test_scaled_tensor)
    loss = criterion(preds, y_test_tensor).item()

    accuracy = ((preds >= 0.5) == y_test_tensor).float().mean().item()

In [42]:
accuracy

0.9824561476707458